# CKP02 — DocMind RAG | Elytra Energy — versão otimizada
FIAP · Prompt Engineering and Artificial Intelligence · Prof. Jorge Luiz Gomes · 2º semestre de 2026

**Integrantes:** Kaique Da Silva Assis (RM 572718) · Andre Debiazzi (RM 569062) · Vinicius Cristal de Oliveira (RM 572049)

**Modo rápido para concluir a execução:** usa as cinco fontes oficiais, compara duas configurações válidas (chunk 512 e 1024), processa lotes de embeddings maiores, grava embeddings em cache no Google Drive para retomar após desconexões e reutiliza índices completos. Para manter a execução rápida, o padrão indexa as primeiras 15 páginas de cada PDF; os arquivos PDF completos continuam na base. Para indexar todas as páginas, altere `RAG_MAX_PAGES_PER_PDF` para `0` na célula 0 (isso poderá demorar muito mais).

**Antes de executar:** em *Ambiente de execução → Alterar tipo de ambiente de execução*, selecione **T4 GPU** se estiver disponível. A primeira célula pedirá autorização para acessar uma pasta específica do Google Drive (`MyDrive/CKP02_DocMind_RAG_cache`); ela guarda PDFs e embeddings processados entre sessões.

Pipeline: `load → split → embed → store → retrieve → generate`, ChromaDB, comparação de chunking com RAGAS, metadata filtering e interface Gradio. O reranker continua implementado, mas fica desligado por padrão para evitar baixar outro modelo pesado no caminho crítico; habilite-o na célula 0 se sobrar tempo.

## 0. Preparação do ambiente (dependências, Secret e Ollama na VM)

In [1]:
import os, subprocess, sys, time, urllib.request, shutil
from pathlib import Path

# O cache no Drive preserva embeddings já calculados mesmo se o runtime for desconectado.
from google.colab import drive, userdata
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
else:
    print("Google Drive já está montado.")

# Carrega o Secret sem expor a chave no notebook/saída.
try:
    chave = userdata.get("OLLAMA_API_KEY")
except Exception as exc:
    raise RuntimeError("Crie o Secret OLLAMA_API_KEY no painel Secrets e habilite Acesso ao notebook.") from exc
if not chave:
    raise RuntimeError("O Secret OLLAMA_API_KEY está vazio.")
os.environ["OLLAMA_API_KEY"] = chave

CACHE_ROOT = Path("/content/drive/MyDrive/CKP02_DocMind_RAG_cache")
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["EMBEDDING_CACHE_DIR"] = str(CACHE_ROOT / "embedding_batches")
os.environ["RAG_PERSISTENT_DOCS_DIR"] = str(CACHE_ROOT / "originals")
os.environ["OLLAMA_CLOUD_HOST"] = "https://ollama.com"
os.environ["OLLAMA_CHAT_MODEL"] = "gemma4:cloud"          # modelo pedido no enunciado
os.environ["OLLAMA_EMBEDDING_MODEL"] = "nomic-embed-text"  # único modelo aprovado
os.environ["OLLAMA_EMBEDDING_MODE"] = "auto"               # tenta Cloud; fallback para Ollama na VM
os.environ["ENABLE_RERANKER"] = "false"                    # opcional; pode habilitar após indexar
os.environ["RAG_MAX_PAGES_PER_PDF"] = "15"                  # modo rápido; 0 = todas as páginas

try:
    gpu = subprocess.run(["nvidia-smi", "-L"], check=True, capture_output=True, text=True)
    print("GPU disponível:", gpu.stdout.strip().replace("\n", "; "))
except Exception:
    print("GPU NVIDIA não detectada. Se disponível, escolha T4 GPU antes de executar a célula 0.")

# Instala apenas dependências usadas pelo notebook; reranker opcional não instala sentence-transformers.
print("Instalando dependências Python...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "langchain-core", "langchain-text-splitters", "langchain-ollama", "langchain-chroma",
                "langchain-openai", "chromadb", "pydantic", "python-dotenv", "pypdf", "requests",
                "gradio", "ragas>=0.2.10,<0.4", "datasets", "pytest"], check=True)

# Instala o servidor Ollama apenas se necessário.
if shutil.which("ollama") is None:
    print("Instalando Ollama na VM do Colab...")
    subprocess.run("apt-get install -y -qq zstd pciutils > /dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1",
                   shell=True, check=True)
else:
    print("Ollama já está instalado.")

try:
    urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2).close()
    print("Servidor Ollama já está respondendo.")
except Exception:
    subprocess.Popen("nohup ollama serve > /tmp/ollama.log 2>&1 &", shell=True)
    for _ in range(60):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2).close()
            print("Servidor Ollama iniciado.")
            break
        except Exception:
            time.sleep(1)
    else:
        print(open("/tmp/ollama.log").read()[-2000:])
        raise RuntimeError("O servidor Ollama não iniciou. Veja o log acima.")

# Não baixa 274 MB novamente caso o modelo já esteja na VM atual.
lista = subprocess.run(["ollama", "list"], capture_output=True, text=True, check=True).stdout
if "nomic-embed-text" not in lista:
    print("Baixando nomic-embed-text (necessário nesta nova VM)...")
    subprocess.run(["ollama", "pull", "nomic-embed-text"], check=True)
else:
    print("Modelo nomic-embed-text já está baixado.")
print("Ambiente pronto. Cache persistente:", os.environ["EMBEDDING_CACHE_DIR"])

Mounted at /content/drive
GPU disponível: GPU 0: Tesla T4 (UUID: GPU-db46c46c-e2ff-fa50-9e0c-a98e4d2916da)
Instalando dependências Python...
Instalando Ollama na VM do Colab...
Servidor Ollama iniciado.
Baixando nomic-embed-text (necessário nesta nova VM)...
Ambiente pronto. Cache persistente: /content/drive/MyDrive/CKP02_DocMind_RAG_cache/embedding_batches


## 1. Código do projeto (módulos em `src/`, `scripts/`, `evals/`, `tests/`)

In [2]:
!mkdir -p /content/CKP02_DocMind_RAG_Elytra_Energy/src /content/CKP02_DocMind_RAG_Elytra_Energy/scripts /content/CKP02_DocMind_RAG_Elytra_Energy/evals /content/CKP02_DocMind_RAG_Elytra_Energy/tests /content/CKP02_DocMind_RAG_Elytra_Energy/data/originals /content/CKP02_DocMind_RAG_Elytra_Energy/prompts /content/CKP02_DocMind_RAG_Elytra_Energy/reports /content/CKP02_DocMind_RAG_Elytra_Energy/chroma_db

In [3]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/src/__init__.py

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/src/__init__.py


In [4]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/src/config.py
"""Configuração central do DocMind RAG — Elytra Energy (CKP02).

Todas as constantes do projeto ficam aqui para que o restante do código seja
modular e para que o pipeline possa ser reutilizado como @tool no CKP03.
"""
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv

ROOT_DIR = Path(__file__).resolve().parent.parent
load_dotenv(ROOT_DIR / ".env")

# ----------------------------------------------------------------------------
# Pastas e arquivos
# ----------------------------------------------------------------------------
DATA_DIR = ROOT_DIR / "data" / "originals"
SOURCES_FILE = ROOT_DIR / "data" / "fontes.json"
QUESTIONS_FILE = ROOT_DIR / "evals" / "perguntas.json"
CHROMA_DIR = ROOT_DIR / "chroma_db"
REPORTS_DIR = ROOT_DIR / "reports"
PROMPT_FILE = ROOT_DIR / "prompts" / "system_prompt.md"

# ----------------------------------------------------------------------------
# Modelo de chat — exigência do PDF: exclusivamente gemma4:cloud (Ollama Cloud)
# ----------------------------------------------------------------------------
# "gemma4:cloud" é o nome pedido no enunciado. A API HTTP direta do Ollama Cloud
# usa o nome devolvido por https://ollama.com/api/tags (hoje "gemma4:31b").
# O módulo generator.py resolve o nome automaticamente; veja docs/DECISOES_TECNICAS.md.
CHAT_MODEL = os.getenv("OLLAMA_CHAT_MODEL", "gemma4:cloud").strip()
OLLAMA_CLOUD_HOST = os.getenv("OLLAMA_CLOUD_HOST", "https://ollama.com").rstrip("/")
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY", "").strip()  # Secret; nunca no código
CHAT_TEMPERATURE = 0  # exigido pelo PDF: respostas fundamentadas

# ----------------------------------------------------------------------------
# Embeddings — exigência do PDF: nomic-embed-text com OllamaEmbeddings
# ----------------------------------------------------------------------------
EMBEDDING_MODEL = os.getenv("OLLAMA_EMBEDDING_MODEL", "nomic-embed-text").strip()
# auto  -> tenta o endpoint de embeddings do Ollama Cloud; se não houver suporte
#          (HTTP 401/403/404), usa o Ollama do runtime (Colab) ou da máquina.
# cloud -> exige o Ollama Cloud. local -> exige o Ollama em EMBEDDING_LOCAL_HOST.
EMBEDDING_MODE = os.getenv("OLLAMA_EMBEDDING_MODE", "auto").strip().lower()
EMBEDDING_LOCAL_HOST = os.getenv("OLLAMA_EMBEDDING_HOST", "http://127.0.0.1:11434").rstrip("/")

# ----------------------------------------------------------------------------
# RAG
# ----------------------------------------------------------------------------
CHUNK_SIZES = (256, 512, 1024)          # PDF: comparar no mínimo 2 configurações
OVERLAP_RATIO = 0.12                    # PDF: overlap de 10% a 15% do chunk_size
DEFAULT_CHUNK_SIZE = int(os.getenv("RAG_CHUNK_SIZE", "512"))
TOP_K = int(os.getenv("RAG_TOP_K", "5"))
ENABLE_RERANKER = os.getenv("ENABLE_RERANKER", "true").strip().lower() in {"1", "true", "yes", "sim"}
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"  # citado no PDF (Aula 07)

# Nome da coleção = nome do domínio (exigência do PDF) + tamanho do chunk.
COLLECTION_PREFIX = "eficiencia_energetica_residencial"

CATEGORIAS = [
    "Climatização", "Refrigeração", "Ventilação",
    "Residencial", "Indicadores energéticos",
]


def collection_name(chunk_size: int) -> str:
    """Devolve o nome da coleção ChromaDB para cada configuração de chunking."""
    if chunk_size not in CHUNK_SIZES:
        raise ValueError(f"chunk_size inválido: {chunk_size}. Use um de {CHUNK_SIZES}.")
    return f"{COLLECTION_PREFIX}_{chunk_size}"

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/src/config.py


In [5]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/src/schemas.py
"""Schemas Pydantic: metadados, chunks e resposta com proveniência (fonte/página/chunk)."""
from __future__ import annotations

from pydantic import BaseModel, ConfigDict, Field, field_validator


class DocumentMetadata(BaseModel):
    model_config = ConfigDict(extra="forbid")
    arquivo: str = Field(min_length=1)
    tipo: str = Field(min_length=1)
    categoria: str = Field(min_length=1)
    data: str = Field(min_length=4)
    fonte: str = Field(min_length=1)
    instituicao: str = Field(default="Não informada", min_length=1)


class DocumentRecord(BaseModel):
    model_config = ConfigDict(extra="forbid")
    conteudo: str = Field(min_length=1)
    metadata: DocumentMetadata

    @field_validator("conteudo")
    @classmethod
    def conteudo_nao_pode_ser_vazio(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("O conteúdo do documento não pode estar vazio.")
        return value


class ChunkRecord(BaseModel):
    model_config = ConfigDict(extra="forbid")
    id: str = Field(min_length=1)
    conteudo: str = Field(min_length=1)
    arquivo: str = Field(min_length=1)
    pagina: int = Field(ge=1)
    categoria: str = Field(min_length=1)
    fonte: str = Field(min_length=1)


class SourceCitation(BaseModel):
    """Citação do chunk de origem (exigência do PDF: a resposta cita o chunk)."""
    model_config = ConfigDict(extra="forbid")
    trecho_numero: int = Field(default=1, ge=1)
    arquivo: str = Field(min_length=1)
    pagina: int = Field(ge=1)
    fonte: str = Field(min_length=1)
    categoria: str = Field(default="Geral", min_length=1)
    chunk_id: str = Field(default="")
    trecho: str = Field(default="")  # início do texto do chunk recuperado


class RAGResponse(BaseModel):
    model_config = ConfigDict(extra="forbid")
    consulta: str = Field(min_length=1)
    resposta: str = Field(min_length=1)
    fontes: list[SourceCitation] = Field(default_factory=list)
    contextos: list[str] = Field(default_factory=list)  # textos enviados ao LLM (usados no RAGAS)
    chunks_recuperados: int = Field(default=0, ge=0)
    configuracao_chunking: int = Field(default=512, ge=1)

    def para_texto(self) -> str:
        """Resposta + fontes rastreáveis (arquivo, página, chunk e URL de origem)."""
        if not self.fontes:
            return self.resposta + "\n\n**Fontes consultadas:** nenhum trecho relevante foi recuperado."
        linhas = [self.resposta, "", "**Fontes consultadas**"]
        for f in self.fontes:
            resumo = " ".join(f.trecho.split())[:160]
            linhas.append(
                f"- [Trecho {f.trecho_numero}] `{f.arquivo}`, página {f.pagina} "
                f"(chunk `{f.chunk_id}`) — {f.categoria} — {f.fonte}"
                + (f"\n  > {resumo}…" if resumo else "")
            )
        return "\n".join(linhas)

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/src/schemas.py


In [6]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/src/loaders.py
"""Carrega PDFs, TXT e Markdown da base e anexa metadata rastreável (tipo, data, categoria).

Usa `pypdf` diretamente (sem langchain_community) para reduzir dependências e
evitar incompatibilidades entre versões do LangChain.
"""
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

from langchain_core.documents import Document
from pypdf import PdfReader

from .config import DATA_DIR, SOURCES_FILE
from .schemas import DocumentMetadata, DocumentRecord

EXTENSOES = {".pdf", ".txt", ".md", ".markdown"}


def _manifesto() -> dict[str, dict[str, Any]]:
    """Lê data/fontes.json (fonte, data, categoria, tipo, instituição de cada arquivo)."""
    if not SOURCES_FILE.exists():
        return {}
    payload = json.loads(SOURCES_FILE.read_text(encoding="utf-8"))
    return payload.get("documentos", {})


def _categoria_padrao(nome: str) -> str:
    """Categoria deduzida do nome do arquivo quando ele não está no manifesto."""
    nome = nome.lower()
    if "condicionador" in nome or "climat" in nome:
        return "Climatização"
    if "refrigerador" in nome or "geladeira" in nome:
        return "Refrigeração"
    if "ventilador" in nome:
        return "Ventilação"
    if "atlas" in nome or "epe" in nome:
        return "Indicadores energéticos"
    return "Residencial"


def _paginas_pdf(path: Path, max_paginas: int | None) -> list[tuple[int, str]]:
    """Extrai (número da página começando em 1, texto) de cada página do PDF."""
    leitor = PdfReader(str(path))
    if leitor.is_encrypted:
        try:
            leitor.decrypt("")
        except Exception as exc:  # PDF protegido por senha real
            raise RuntimeError(f"PDF protegido por senha: {path.name}") from exc
    total = len(leitor.pages)
    limite = min(total, max_paginas) if max_paginas else total
    paginas = []
    for indice in range(limite):
        try:
            texto = leitor.pages[indice].extract_text() or ""
        except Exception:
            texto = ""  # página com erro de extração: ignora e segue
        paginas.append((indice + 1, texto))
    return paginas


def _carregar_arquivo(path: Path) -> list[Document]:
    ext = path.suffix.lower()
    registro_fonte = _manifesto().get(path.name, {})
    max_paginas = registro_fonte.get("max_paginas")
    # Modo rápido: limita páginas por arquivo; 0 significa todas as páginas.
    limite_global = int(os.getenv("RAG_MAX_PAGES_PER_PDF", "20"))
    if limite_global > 0:
        max_paginas = min(int(max_paginas), limite_global) if max_paginas else limite_global
    if ext == ".pdf":
        paginas = _paginas_pdf(path, int(max_paginas) if max_paginas else None)
    elif ext in {".txt", ".md", ".markdown"}:
        paginas = [(1, path.read_text(encoding="utf-8", errors="replace"))]
    else:
        return []

    fonte = str(registro_fonte.get("fonte") or path.resolve().as_uri())
    categoria = str(registro_fonte.get("categoria") or _categoria_padrao(path.name))
    data = str(registro_fonte.get("data") or "Não informada")
    tipo = str(registro_fonte.get("tipo") or ext.removeprefix(".").upper())
    instituicao = str(registro_fonte.get("instituicao") or "Não informada")

    documentos: list[Document] = []
    for pagina, texto in paginas:
        conteudo = texto.strip()
        if len(conteudo) < 20:  # páginas vazias ou só com número
            continue
        # Valida conteúdo e metadata com Pydantic antes de entrar no RAG.
        metadata = DocumentMetadata(
            arquivo=path.name, tipo=tipo, categoria=categoria,
            data=data, fonte=fonte, instituicao=instituicao,
        )
        registro = DocumentRecord(conteudo=conteudo, metadata=metadata)
        meta = registro.metadata.model_dump()
        meta["pagina"] = max(1, pagina)
        documentos.append(Document(page_content=registro.conteudo, metadata=meta))
    return documentos


def carregar_documentos(diretorio: Path | None = None) -> list[Document]:
    """Carrega todos os documentos suportados; falha cedo se a base estiver vazia."""
    origem = diretorio or DATA_DIR
    if not origem.exists():
        raise FileNotFoundError(
            f"Pasta da base não encontrada: {origem}. Execute `python -m scripts.baixar_fontes`."
        )
    arquivos = sorted(p for p in origem.rglob("*") if p.is_file() and p.suffix.lower() in EXTENSOES)
    if not arquivos:
        raise FileNotFoundError(
            f"Nenhum PDF/TXT/Markdown em {origem}. Execute `python -m scripts.baixar_fontes`."
        )
    documentos: list[Document] = []
    erros: list[str] = []
    for arquivo in arquivos:
        try:
            documentos.extend(_carregar_arquivo(arquivo))
        except Exception as exc:
            erros.append(f"{arquivo.name}: {exc}")
    if erros:
        raise RuntimeError("Falha ao carregar documentos:\n" + "\n".join(f"- {e}" for e in erros))
    if not documentos:
        raise ValueError("Os documentos existem, mas nenhum texto legível foi extraído (PDF escaneado?).")
    return documentos

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/src/loaders.py


In [7]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/src/chunking.py
"""Chunking exigido pelo PDF: RecursiveCharacterTextSplitter com os separadores da Aula 06
e chunk_overlap entre 10% e 15% do chunk_size (usamos 12%)."""
from __future__ import annotations

from dataclasses import dataclass

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from .config import CHUNK_SIZES, OVERLAP_RATIO
from .schemas import ChunkRecord

# Separadores exatamente como no enunciado.
SEPARADORES = ["\n\n", "\n", ". ", " ", ""]


@dataclass(frozen=True)
class ChunkConfig:
    chunk_size: int
    chunk_overlap: int


def config_chunking(chunk_size: int) -> ChunkConfig:
    """Devolve a configuração (tamanho + overlap de 12%) para um tamanho válido."""
    if chunk_size not in CHUNK_SIZES:
        raise ValueError(f"Tamanho inválido {chunk_size}. Valores aceitos: {CHUNK_SIZES}.")
    return ChunkConfig(chunk_size=chunk_size, chunk_overlap=round(chunk_size * OVERLAP_RATIO))


def criar_splitter(chunk_size: int) -> RecursiveCharacterTextSplitter:
    config = config_chunking(chunk_size)
    return RecursiveCharacterTextSplitter(
        chunk_size=config.chunk_size,
        chunk_overlap=config.chunk_overlap,
        separators=SEPARADORES,
        length_function=len,
        add_start_index=True,
    )


def dividir_documentos(documentos: list[Document], chunk_size: int) -> list[Document]:
    """Divide as páginas em chunks, completa a metadata e valida arquivo/página via Pydantic."""
    if not documentos:
        raise ValueError("Não há documentos para dividir em chunks.")

    chunks = criar_splitter(chunk_size).split_documents(documentos)
    for indice, chunk in enumerate(chunks):
        meta = chunk.metadata
        meta.setdefault("arquivo", "documento_sem_nome")
        meta.setdefault("fonte", "Fonte não informada")
        meta.setdefault("categoria", "Geral")
        meta.setdefault("tipo", "Desconhecido")
        meta.setdefault("data", "Não informada")
        try:
            pagina = max(1, int(meta.get("pagina", 1)))
        except (TypeError, ValueError):
            pagina = 1
        meta["pagina"] = pagina
        meta["chunk_indice"] = indice
        meta["chunk_size_configurado"] = chunk_size
        registro = ChunkRecord(
            id=f"{meta['arquivo']}:{pagina}:{indice}:{chunk_size}",
            conteudo=chunk.page_content,
            arquivo=str(meta["arquivo"]),
            pagina=pagina,
            categoria=str(meta["categoria"]),
            fonte=str(meta["fonte"]),
        )
        meta["chunk_id"] = registro.id
    return chunks

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/src/chunking.py


In [8]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/src/embeddings.py
"""Embeddings: OllamaEmbeddings(model="nomic-embed-text") — único modelo aprovado no PDF.

Estratégia (modo `auto`, padrão):
1. Tenta o endpoint de embeddings do Ollama Cloud (https://ollama.com/api/embed)
   com a chave OLLAMA_API_KEY.
2. Se o Cloud não atender (HTTP 401/403/404 — o que aconteceu no desenvolvimento),
   usa o servidor Ollama do runtime (Colab) ou da máquina, em OLLAMA_EMBEDDING_HOST.

O modelo, a classe e as chamadas (embed_query / embed_documents) são sempre os do enunciado.
"""
from __future__ import annotations

from functools import lru_cache

import requests
from langchain_ollama import OllamaEmbeddings

from .config import (
    EMBEDDING_LOCAL_HOST, EMBEDDING_MODE, EMBEDDING_MODEL,
    OLLAMA_API_KEY, OLLAMA_CLOUD_HOST,
)


def _cabecalhos_cloud() -> dict[str, str]:
    return {"Authorization": f"Bearer {OLLAMA_API_KEY}"} if OLLAMA_API_KEY else {}


def _cloud_atende_embeddings() -> tuple[bool, str]:
    """Testa uma chamada real de embedding no Ollama Cloud. Devolve (ok, detalhe)."""
    if not OLLAMA_API_KEY:
        return False, "OLLAMA_API_KEY ausente"
    try:
        resp = requests.post(
            f"{OLLAMA_CLOUD_HOST}/api/embed",
            json={"model": EMBEDDING_MODEL, "input": "teste de embedding"},
            headers=_cabecalhos_cloud(), timeout=30,
        )
    except requests.RequestException as exc:
        return False, f"erro de rede: {exc}"
    if resp.status_code == 200 and resp.json().get("embeddings"):
        return True, "HTTP 200"
    return False, f"HTTP {resp.status_code}"


def _local_disponivel() -> tuple[bool, str]:
    """Verifica se há Ollama local/Colab com o nomic-embed-text baixado."""
    try:
        resp = requests.get(f"{EMBEDDING_LOCAL_HOST}/api/tags", timeout=5)
        resp.raise_for_status()
    except requests.RequestException as exc:
        return False, f"Ollama não responde em {EMBEDDING_LOCAL_HOST} ({exc})"
    nomes = [m.get("name", "") for m in resp.json().get("models", [])]
    if not any(n.split(":")[0] == EMBEDDING_MODEL for n in nomes):
        return False, f"modelo '{EMBEDDING_MODEL}' não baixado; execute `ollama pull {EMBEDDING_MODEL}`"
    return True, "ok"


@lru_cache(maxsize=1)
def modo_embeddings_efetivo() -> str:
    """Decide (uma única vez) entre 'cloud' e 'local' e informa o motivo na tela."""
    if EMBEDDING_MODE == "cloud":
        ok, detalhe = _cloud_atende_embeddings()
        if not ok:
            raise RuntimeError(f"Embeddings via Ollama Cloud indisponíveis: {detalhe}.")
        return "cloud"
    if EMBEDDING_MODE == "auto":
        ok, detalhe = _cloud_atende_embeddings()
        if ok:
            print("[embeddings] Usando nomic-embed-text no Ollama Cloud.")
            return "cloud"
        print(f"[embeddings] Ollama Cloud não atendeu embeddings ({detalhe}); usando Ollama do runtime.")
    ok, detalhe = _local_disponivel()
    if not ok:
        raise RuntimeError(
            f"Embeddings indisponíveis: {detalhe}. No Colab, execute a célula de preparação; "
            "localmente, inicie o Ollama e rode `ollama pull nomic-embed-text`."
        )
    return "local"


def criar_embeddings() -> OllamaEmbeddings:
    """Retorna OllamaEmbeddings(model='nomic-embed-text') apontando para Cloud ou runtime."""
    if modo_embeddings_efetivo() == "cloud":
        return OllamaEmbeddings(
            model=EMBEDDING_MODEL, base_url=OLLAMA_CLOUD_HOST,
            client_kwargs={"headers": _cabecalhos_cloud()},
        )
    return OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=EMBEDDING_LOCAL_HOST)

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/src/embeddings.py


In [9]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/src/vectorstore.py
"""ChromaDB persistente com cache retomável dos embeddings por lote no Google Drive."""
from __future__ import annotations

import hashlib
import os
import time
from pathlib import Path

import chromadb
import numpy as np
from langchain_chroma import Chroma
from langchain_core.documents import Document

from .config import CHROMA_DIR, EMBEDDING_MODEL, collection_name
from .embeddings import criar_embeddings

TAMANHO_LOTE = 96  # mais eficiência por chamada; reduz overhead comparado a lotes pequenos


def _nomes_colecoes(client) -> set[str]:
    return {item.name if hasattr(item, "name") else str(item) for item in client.list_collections()}


def _ids_deterministicos(chunks: list[Document]) -> list[str]:
    ids = []
    for indice, doc in enumerate(chunks):
        bruto = "|".join([
            str(doc.metadata.get("arquivo", "")),
            str(doc.metadata.get("pagina", 1)),
            str(indice),
            doc.page_content,
        ])
        ids.append(hashlib.sha256(bruto.encode("utf-8")).hexdigest())
    return ids


def _cache_path(chunk_size: int, batch_ids: list[str]) -> Path | None:
    pasta = os.getenv("EMBEDDING_CACHE_DIR", "").strip()
    if not pasta:
        return None
    try:
        destino = Path(pasta)
        destino.mkdir(parents=True, exist_ok=True)
    except OSError as exc:
        print(f"[cache] Não foi possível acessar o cache persistente: {exc}", flush=True)
        return None
    assinatura = hashlib.sha256(
        (f"v2|{EMBEDDING_MODEL}|{chunk_size}|" + "|".join(batch_ids)).encode("utf-8")
    ).hexdigest()
    return destino / f"{chunk_size}_{assinatura}.npz"


def construir_vectorstore(chunks: list[Document], chunk_size: int, substituir: bool = True) -> Chroma:
    """Gera/recupera embeddings por lote e grava a coleção no ChromaDB local."""
    if not chunks:
        raise ValueError("Não é possível criar um índice vazio.")
    CHROMA_DIR.mkdir(parents=True, exist_ok=True)
    nome = collection_name(chunk_size)
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    existentes = _nomes_colecoes(client)
    embeddings = criar_embeddings()

    if nome in existentes:
        atual = Chroma(collection_name=nome, embedding_function=embeddings,
                       persist_directory=str(CHROMA_DIR))
        if not substituir and atual._collection.count() == len(chunks):
            print(f"[reutilizado] {nome}: {len(chunks)} chunks já indexados.", flush=True)
            return atual
        client.delete_collection(name=nome)

    store = Chroma(collection_name=nome, embedding_function=embeddings,
                   persist_directory=str(CHROMA_DIR))
    ids = _ids_deterministicos(chunks)
    total = len(chunks)
    lotes = (total + TAMANHO_LOTE - 1) // TAMANHO_LOTE
    hits_cache = 0
    inicio_total = time.time()

    for numero_lote, inicio in enumerate(range(0, total, TAMANHO_LOTE), start=1):
        fim = min(inicio + TAMANHO_LOTE, total)
        lote = chunks[inicio:fim]
        lote_ids = ids[inicio:fim]
        textos = [doc.page_content for doc in lote]
        metadados = [doc.metadata for doc in lote]
        arquivo_cache = _cache_path(chunk_size, lote_ids)
        vetores = None

        usou_cache = False
        if arquivo_cache is not None and arquivo_cache.is_file():
            try:
                with np.load(arquivo_cache) as salvo:
                    matriz = salvo["embeddings"]
                if matriz.shape[0] == len(lote):
                    vetores = matriz.tolist()
                    hits_cache += 1
                    usou_cache = True
            except Exception:
                vetores = None  # cache corrompido: calcula novamente

        if vetores is None:
            vetores = embeddings.embed_documents(textos)
            if arquivo_cache is not None:
                try:
                    temporario = arquivo_cache.with_name(arquivo_cache.stem + ".tmp.npz")
                    with temporario.open("wb") as f:
                        np.savez_compressed(f, embeddings=np.asarray(vetores, dtype=np.float32))
                    os.replace(temporario, arquivo_cache)
                except OSError as exc:
                    print(f"[cache] Falha ao salvar lote (o índice segue normalmente): {exc}", flush=True)

        store._collection.add(ids=lote_ids, embeddings=vetores,
                              documents=textos, metadatas=metadados)
        elapsed = time.time() - inicio_total
        origem = "cache" if usou_cache else "embedding calculado"
        print(f"  [{nome}] {fim}/{total} chunks | lote {numero_lote}/{lotes} | cache usados: {hits_cache}/{numero_lote} | {elapsed/60:.1f} min | {origem}", flush=True)

    print(f"[concluído] {nome}: {store._collection.count()} registros em {(time.time()-inicio_total)/60:.1f} min.", flush=True)
    return store


def carregar_vectorstore(chunk_size: int) -> Chroma:
    """Abre coleção pronta; falha com mensagem clara se ela não existir ou estiver vazia."""
    CHROMA_DIR.mkdir(parents=True, exist_ok=True)
    nome = collection_name(chunk_size)
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    if nome not in _nomes_colecoes(client):
        raise FileNotFoundError(
            f"Índice '{nome}' não existe. Execute `python -m scripts.build_index --chunk-size {chunk_size}`."
        )
    store = Chroma(collection_name=nome, embedding_function=criar_embeddings(),
                   persist_directory=str(CHROMA_DIR))
    if store._collection.count() == 0:
        raise ValueError(f"A coleção '{nome}' existe, mas está vazia. Reconstrua o índice.")
    return store

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/src/vectorstore.py


In [10]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/src/reranker.py
"""Reranking com cross-encoder (diferencial do PDF: cross-encoder/ms-marco-MiniLM-L-6-v2).

Se o `sentence-transformers` ou o download do modelo falharem, o pipeline
continua sem reranking (e avisa uma única vez) em vez de quebrar a execução.
"""
from __future__ import annotations

from functools import lru_cache

from langchain_core.documents import Document

from .config import RERANKER_MODEL

_falhou = False  # evita repetir o aviso e novas tentativas de download


@lru_cache(maxsize=1)
def _modelo():
    from sentence_transformers import CrossEncoder
    return CrossEncoder(RERANKER_MODEL, device="cpu")


def reranquear(consulta: str, documentos: list[Document], habilitado: bool = True) -> list[Document]:
    """Reordena os candidatos pela pontuação do cross-encoder (maior relevância primeiro)."""
    global _falhou
    if not habilitado or _falhou or len(documentos) < 2:
        return documentos
    try:
        pares = [(consulta, doc.page_content[:2000]) for doc in documentos]
        pontuacoes = _modelo().predict(pares, show_progress_bar=False)
    except Exception as exc:  # sem a lib, sem internet para baixar os pesos etc.
        _falhou = True
        print(f"[reranker] Desativado nesta execução ({type(exc).__name__}: {exc}).")
        return documentos
    ordenados = sorted(zip(documentos, pontuacoes), key=lambda par: float(par[1]), reverse=True)
    return [doc for doc, _ in ordenados]

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/src/reranker.py


In [11]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/src/retriever.py
"""Recuperação semântica (retrieve) com metadata filtering e reranking (diferenciais do PDF)."""
from __future__ import annotations

from langchain_chroma import Chroma
from langchain_core.documents import Document

from .config import CATEGORIAS, ENABLE_RERANKER, TOP_K
from .reranker import reranquear


def montar_filtro(categoria: str | None = None, instituicao: str | None = None) -> dict | None:
    """Monta o `where` do Chroma a partir dos metadados (categoria e/ou instituição)."""
    condicoes = []
    if categoria and categoria != "Todas":
        if categoria not in CATEGORIAS:
            raise ValueError(f"Categoria não reconhecida: {categoria}. Use uma de {CATEGORIAS}.")
        condicoes.append({"categoria": categoria})
    if instituicao and instituicao != "Todas":
        condicoes.append({"instituicao": instituicao})
    if not condicoes:
        return None
    return condicoes[0] if len(condicoes) == 1 else {"$and": condicoes}


def recuperar(
    vectorstore: Chroma,
    consulta: str,
    categoria: str | None = None,
    instituicao: str | None = None,
    k: int = TOP_K,
    usar_reranking: bool | None = None,
) -> list[Document]:
    """Busca os k chunks mais relevantes; com reranking, busca mais candidatos e reordena."""
    if not consulta.strip():
        raise ValueError("A consulta de recuperação não pode estar vazia.")
    if k < 1:
        raise ValueError("k precisa ser maior que zero.")

    reranking_ativo = ENABLE_RERANKER if usar_reranking is None else usar_reranking
    kwargs: dict = {"k": max(k * 4, 20) if reranking_ativo else k}
    filtro = montar_filtro(categoria, instituicao)
    if filtro:
        kwargs["filter"] = filtro  # o langchain-chroma converte em `where=` do ChromaDB
    candidatos = vectorstore.similarity_search(consulta, **kwargs)
    ordenados = reranquear(consulta, candidatos, habilitado=reranking_ativo)
    return ordenados[:k]

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/src/retriever.py


In [12]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/src/generator.py
"""Geração com modelo Gemma no endpoint OpenAI-compatible oficial do Ollama Cloud."""
from __future__ import annotations

import time
from functools import lru_cache

import requests
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

from .config import CHAT_MODEL, CHAT_TEMPERATURE, OLLAMA_API_KEY, OLLAMA_CLOUD_HOST, PROMPT_FILE


@lru_cache(maxsize=1)
def _prompt() -> ChatPromptTemplate:
    instrucoes = PROMPT_FILE.read_text(encoding="utf-8")
    return ChatPromptTemplate.from_messages([
        ("system", "{instrucoes}"),
        ("human",
         "Pergunta do usuário: {question}\n\n"
         "Trechos recuperados da base oficial:\n{context}\n\n"
         "Responda somente com base nesses trechos, citando [Trecho N]. "
         "Se a evidência for insuficiente, diga isso claramente."),
    ]).partial(instrucoes=instrucoes)


@lru_cache(maxsize=1)
def resolver_nome_modelo() -> str:
    """Escolhe uma variante Gemma retornada pela API do Ollama Cloud."""
    if not OLLAMA_API_KEY:
        raise EnvironmentError("OLLAMA_API_KEY ausente. Configure o Secret no Colab.")
    try:
        resp = requests.get(
            f"{OLLAMA_CLOUD_HOST}/api/tags",
            headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"}, timeout=20,
        )
        resp.raise_for_status()
        nomes = [m.get("name") or m.get("model", "") for m in resp.json().get("models", [])]
    except requests.RequestException as exc:
        raise RuntimeError(f"Não foi possível consultar modelos do Ollama Cloud: {exc}") from exc
    base = CHAT_MODEL.removesuffix(":cloud").removesuffix("-cloud")
    candidatos = [CHAT_MODEL, base, f"{base}:31b", f"{base}:31b-cloud"]
    for nome in candidatos:
        if nome in nomes:
            return nome
    variantes = sorted(n for n in nomes if n.split(":")[0] == base)
    return variantes[-1] if variantes else f"{base}:31b"


def criar_modelo_chat() -> ChatOpenAI:
    """Cliente autenticado para https://ollama.com/v1; evita o 401 do ChatOllama nesse endpoint."""
    return ChatOpenAI(
        model=resolver_nome_modelo(),
        api_key=OLLAMA_API_KEY,
        base_url=f"{OLLAMA_CLOUD_HOST}/v1",
        temperature=CHAT_TEMPERATURE,
        timeout=90,
        max_retries=2,
    )


def _texto(resposta) -> str:
    conteudo = resposta.content
    if isinstance(conteudo, str):
        return conteudo.strip()
    return "\n".join(str(item) for item in conteudo).strip()


def gerar_resposta(pergunta: str, contexto: str, modelo=None, tentativas: int = 2) -> str:
    """Gera resposta fundamentada; usa poucas tentativas para evitar esperas longas."""
    if not pergunta.strip():
        raise ValueError("A pergunta não pode estar vazia.")
    chat = modelo or criar_modelo_chat()
    prompt = _prompt().invoke({"question": pergunta, "context": contexto})
    ultimo_erro: Exception | None = None
    for tentativa in range(1, tentativas + 1):
        try:
            texto = _texto(chat.invoke(prompt))
            if texto:
                return texto
        except Exception as exc:
            ultimo_erro = exc
        if tentativa < tentativas:
            time.sleep(2 * tentativa)
    if ultimo_erro:
        raise RuntimeError(f"O modelo de chat falhou após {tentativas} tentativas: {ultimo_erro}") from ultimo_erro
    return "Não consegui produzir uma resposta com os trechos recuperados."


Writing /content/CKP02_DocMind_RAG_Elytra_Energy/src/generator.py


In [13]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/src/rag.py
"""Pipeline RAG: retrieve → generate, com citação do chunk de origem.

`buscar(consulta)` é a interface pública usada como @tool no CKP03.
"""
from __future__ import annotations

from functools import lru_cache

from langchain_core.documents import Document

from .config import DEFAULT_CHUNK_SIZE, TOP_K
from .generator import criar_modelo_chat, gerar_resposta
from .retriever import recuperar
from .schemas import RAGResponse, SourceCitation
from .vectorstore import carregar_vectorstore


def _contexto(documentos: list[Document]) -> str:
    """Monta o contexto numerado enviado ao LLM ([Trecho N] | arquivo | página)."""
    partes = []
    for i, doc in enumerate(documentos, start=1):
        meta = doc.metadata
        partes.append(
            f"[Trecho {i} | arquivo: {meta.get('arquivo', 'documento sem nome')} "
            f"| página: {int(meta.get('pagina', 1))}]\n{doc.page_content}"
        )
    return "\n\n---\n\n".join(partes)


def _fontes(documentos: list[Document]) -> list[SourceCitation]:
    """Uma citação por chunk recuperado (arquivo, página, chunk_id, URL de origem)."""
    fontes = []
    for i, doc in enumerate(documentos, start=1):
        meta = doc.metadata
        try:
            pagina = max(1, int(meta.get("pagina", 1)))
        except (TypeError, ValueError):
            pagina = 1
        fontes.append(SourceCitation(
            trecho_numero=i,
            arquivo=str(meta.get("arquivo", "documento sem nome")),
            pagina=pagina,
            fonte=str(meta.get("fonte", "Fonte não informada")),
            categoria=str(meta.get("categoria", "Geral")),
            chunk_id=str(meta.get("chunk_id", "")),
            trecho=doc.page_content[:300],
        ))
    return fontes


class RAGPipeline:
    """Recupera evidências no ChromaDB, gera a resposta e anexa a proveniência."""

    def __init__(self, chunk_size: int = DEFAULT_CHUNK_SIZE):
        self.chunk_size = chunk_size
        self.vectorstore = carregar_vectorstore(chunk_size)
        self.modelo_chat = criar_modelo_chat()

    def consultar(self, consulta: str, categoria: str | None = None,
                  instituicao: str | None = None) -> RAGResponse:
        pergunta = consulta.strip()
        if not pergunta:
            raise ValueError("Digite uma pergunta para consultar a base.")
        documentos = recuperar(self.vectorstore, pergunta, categoria=categoria,
                               instituicao=instituicao, k=TOP_K)
        if not documentos:
            return RAGResponse(
                consulta=pergunta,
                resposta="Não encontrei trechos relevantes na base para responder com segurança. "
                         "Tente reformular a pergunta ou remover o filtro de categoria.",
                configuracao_chunking=self.chunk_size,
            )
        resposta = gerar_resposta(pergunta, _contexto(documentos), modelo=self.modelo_chat)
        return RAGResponse(
            consulta=pergunta,
            resposta=resposta,
            fontes=_fontes(documentos),
            contextos=[d.page_content for d in documentos],
            chunks_recuperados=len(documentos),
            configuracao_chunking=self.chunk_size,
        )


@lru_cache(maxsize=3)
def obter_pipeline(chunk_size: int = DEFAULT_CHUNK_SIZE) -> RAGPipeline:
    return RAGPipeline(chunk_size=chunk_size)


def buscar_estruturado(consulta: str, chunk_size: int = DEFAULT_CHUNK_SIZE) -> RAGResponse:
    """Versão que devolve o objeto Pydantic completo (resposta, fontes e contextos)."""
    return obter_pipeline(chunk_size).consultar(consulta)


def buscar(consulta: str) -> str:
    """Consulta a base de eficiência energética e devolve a resposta com as fontes citadas.

    Função pública simples, pronta para virar @tool do agente no CKP03.
    """
    return buscar_estruturado(consulta).para_texto()

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/src/rag.py


In [14]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/scripts/__init__.py

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/scripts/__init__.py


In [15]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/scripts/baixar_fontes.py
"""Baixa os cinco documentos oficiais (PDF) para data/originals/.

Para cada documento, tenta em ordem:
1. as URLs de download diretas conhecidas;
2. a descoberta automática de links .pdf dentro da página oficial de origem;
3. (se tudo falhar) mostra o nome exato do arquivo para você enviar manualmente
   pelo painel de arquivos do Colab / copiar para data/originals/.

Só aceita arquivos que realmente começam com "%PDF-".
"""
from __future__ import annotations

import json
import re
import sys
import time
from pathlib import Path
from urllib.parse import unquote, urljoin

import requests

from src.config import DATA_DIR, SOURCES_FILE

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/124.0 Safari/537.36 ElytraEnergy-CKP02"),
    "Accept": "application/pdf,text/html;q=0.9,*/*;q=0.8",
}

# URLs diretas conhecidas (a página oficial ".../view" é convertida em download).
URLS_DIRETAS = {
    "04_epe_atlas_eficiencia_energetica_2024.pdf": [],
    "05_mme_guia_eficiencia_energetica_edificios_publicos.pdf": [
        "https://www.gov.br/mme/pt-br/centrais-de-conteudo/publicacoes/eficiencia-energetica/guia-de-eficiencia-energetica-em-edificios-publicos.pdf/@@download/file",
    ],
}
# Palavras que identificam o PDF certo ao procurar links dentro de uma página oficial.
PALAVRAS_CHAVE = {
    "01_inmetro_condicionadores_de_ar_2025.pdf": ["condicionadores", "2025"],
    "02_inmetro_refrigeradores_2025.pdf": ["refrigerador"],
    "03_inmetro_ventiladores_2025.pdf": ["ventilador"],
    "04_epe_atlas_eficiencia_energetica_2024.pdf": ["atlas", "2024"],
    "05_mme_guia_eficiencia_energetica_edificios_publicos.pdf": ["guia", "eficiencia"],
}


def _get(url: str, **kwargs) -> requests.Response:
    """GET com fallback para certificados incompletos de sites .gov.br."""
    try:
        return requests.get(url, headers=HEADERS, timeout=(20, 180), **kwargs)
    except requests.exceptions.SSLError:
        print("   [aviso] certificado SSL do site oficial não validou; tentando sem verificação.")
        requests.packages.urllib3.disable_warnings()  # type: ignore[attr-defined]
        return requests.get(url, headers=HEADERS, timeout=(20, 180), verify=False, **kwargs)


def _gravar_pdf(url: str, destino: Path) -> None:
    temporario = destino.with_suffix(".part")
    with _get(url, stream=True) as resposta:
        resposta.raise_for_status()
        with temporario.open("wb") as arquivo:
            for bloco in resposta.iter_content(chunk_size=1024 * 128):
                if bloco:
                    arquivo.write(bloco)
    with temporario.open("rb") as arquivo:
        assinatura = arquivo.read(5)
    if assinatura != b"%PDF-" or temporario.stat().st_size < 5000:
        temporario.unlink(missing_ok=True)
        raise ValueError("a URL não devolveu um PDF válido")
    temporario.replace(destino)


def _candidatas(nome: str, fonte: str) -> list[str]:
    """URLs de download a tentar para um arquivo."""
    urls = list(URLS_DIRETAS.get(nome, []))
    if fonte.endswith(".pdf/view"):
        base = fonte[: -len("/view")]
        urls += [base + "/@@download/file", base]
    elif fonte.lower().endswith(".pdf"):
        urls.append(fonte)
    return list(dict.fromkeys(urls))


def _descobrir_na_pagina(nome: str, pagina: str) -> list[str]:
    """Procura links .pdf na página oficial e escolhe os que casam com as palavras-chave."""
    try:
        html = _get(pagina).text
    except Exception:
        return []
    links = re.findall(r'href=["\']([^"\']+?\.pdf[^"\']*)["\']', html, flags=re.I)
    absolutos = [urljoin(pagina, unquote(link)) for link in links]
    chaves = PALAVRAS_CHAVE.get(nome, [])
    boas = [u for u in absolutos if all(c in u.lower() for c in chaves[:1])]
    return list(dict.fromkeys(boas or absolutos))[:6]


def baixar_documento(nome: str, fonte: str, destino: Path) -> None:
    erros = []
    for url in _candidatas(nome, fonte):
        for tentativa in (1, 2):
            try:
                _gravar_pdf(url, destino)
                return
            except Exception as exc:
                erros.append(f"{url}: {exc}")
                time.sleep(2 * tentativa)
    for url in _descobrir_na_pagina(nome, fonte.removesuffix("/view")):
        try:
            _gravar_pdf(url, destino)
            return
        except Exception as exc:
            erros.append(f"{url}: {exc}")
    raise RuntimeError("; ".join(erros[-3:]) or "nenhuma URL candidata")


def _pdf_valido(caminho: Path) -> bool:
    if not caminho.exists() or caminho.stat().st_size < 5000:
        return False
    with caminho.open("rb") as arquivo:
        return arquivo.read(5) == b"%PDF-"


def main() -> int:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    manifesto = json.loads(SOURCES_FILE.read_text(encoding="utf-8"))["documentos"]
    validos, falhas = 0, []
    for nome, info in manifesto.items():
        destino = DATA_DIR / nome
        if _pdf_valido(destino):
            print(f"[OK] já existe: {nome}")
            validos += 1
            continue
        print(f"[BAIXANDO] {nome}")
        try:
            baixar_documento(nome, info["fonte"], destino)
            print(f"[OK] {nome} ({destino.stat().st_size:,} bytes)")
            validos += 1
        except Exception as exc:
            falhas.append(nome)
            print(f"[ERRO] {nome}: {exc}", file=sys.stderr)
    print(f"\nDocumentos válidos: {validos}/{len(manifesto)}")
    if falhas:
        print("\nNão foi possível baixar automaticamente:")
        for nome in falhas:
            print(f"  - {nome}\n    abra a página oficial: {manifesto[nome]['fonte']}")
        print(f"\nBaixe esses PDFs no navegador, renomeie EXATAMENTE como acima e coloque em:\n  {DATA_DIR}\n"
              "(No Colab: painel de arquivos → pasta data/originals → upload.) Depois execute de novo.")
        return 1
    print(f"Base pronta em: {DATA_DIR}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


Writing /content/CKP02_DocMind_RAG_Elytra_Energy/scripts/baixar_fontes.py


In [16]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/scripts/build_index.py
"""Carrega a base, divide em chunks e grava os índices ChromaDB (uma coleção por chunk size)."""
from __future__ import annotations

import argparse

from src.chunking import config_chunking, dividir_documentos
from src.config import CHUNK_SIZES
from src.loaders import carregar_documentos
from src.vectorstore import construir_vectorstore, carregar_vectorstore


def construir(chunk_size: int, documentos=None) -> int:
    documentos = documentos if documentos is not None else carregar_documentos()
    config = config_chunking(chunk_size)
    chunks = dividir_documentos(documentos, chunk_size)
    try:
        store = carregar_vectorstore(chunk_size)
        if store._collection.count() != len(chunks):
            raise ValueError("Índice incompleto")
        print(f"[REUTILIZADO] chunk_size={chunk_size}: {len(chunks)} chunks.", flush=True)
    except (FileNotFoundError, ValueError):
        store = construir_vectorstore(chunks, chunk_size, substituir=True)
    print(f"chunk_size={chunk_size} | overlap={config.chunk_overlap} ({config.chunk_overlap / chunk_size:.0%}) "
          f"| páginas: {len(documentos)} | chunks: {len(chunks)} | coleção: {store._collection.name}")
    return len(chunks)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--chunk-size", type=int, choices=CHUNK_SIZES, default=512)
    parser.add_argument("--all", action="store_true", help="Constrói as coleções 256, 512 e 1024.")
    args = parser.parse_args()
    documentos = carregar_documentos()
    tamanhos = CHUNK_SIZES if args.all else (args.chunk_size,)
    total = sum(construir(t, documentos) for t in tamanhos)
    print(f"\nConcluído. Total de chunks indexados: {total}.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/scripts/build_index.py


In [17]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/scripts/diagnostico.py
"""Diagnóstico rápido: chave, chat (gemma4:cloud) e embeddings (nomic-embed-text)."""
from __future__ import annotations

from src.config import CHAT_MODEL, EMBEDDING_MODE, EMBEDDING_MODEL, OLLAMA_API_KEY, OLLAMA_CLOUD_HOST


def main() -> int:
    print("=== Diagnóstico DocMind RAG ===")
    print(f"Chave OLLAMA_API_KEY: {'CARREGADA' if OLLAMA_API_KEY else 'AUSENTE'}")
    print(f"Chat: {CHAT_MODEL} em {OLLAMA_CLOUD_HOST} | Embeddings: {EMBEDDING_MODEL} (modo {EMBEDDING_MODE})")
    falhas = 0
    try:
        from src.embeddings import criar_embeddings, modo_embeddings_efetivo
        vetor = criar_embeddings().embed_query("teste de eficiência energética residencial")
        print(f"[OK] Embedding de {len(vetor)} dimensões (origem: {modo_embeddings_efetivo()}).")
    except Exception as exc:
        falhas += 1
        print(f"[FALHA] Embeddings: {type(exc).__name__}: {exc}")
    try:
        from src.generator import criar_modelo_chat, resolver_nome_modelo
        modelo = criar_modelo_chat()
        texto = str(modelo.invoke("Responda apenas: conexão OK").content).strip()
        print(f"[OK] Chat '{resolver_nome_modelo()}' respondeu: {texto[:80]}")
    except Exception as exc:
        falhas += 1
        print(f"[FALHA] Chat Ollama Cloud: {type(exc).__name__}: {exc}")
    print("\nDiagnóstico " + ("INCOMPLETO — corrija os pontos acima." if falhas else "concluído com sucesso."))
    return 1 if falhas else 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/scripts/diagnostico.py


In [18]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/scripts/executar_tudo.py
"""Executa o fluxo otimizado: diagnóstico → fontes → índices 512/1024 → consulta → RAGAS."""
from __future__ import annotations

from scripts import baixar_fontes, diagnostico, build_index


def main() -> int:
    for etapa, funcao in (("Diagnóstico", diagnostico.main), ("Download das fontes", baixar_fontes.main)):
        print(f"\n######## {etapa} ########", flush=True)
        if funcao() != 0:
            print(f"Etapa '{etapa}' falhou; corrija e execute novamente.")
            return 1
    from src.loaders import carregar_documentos
    docs = carregar_documentos()
    print("\n######## Índices ChromaDB (512, 1024) ########", flush=True)
    for tamanho in (512, 1024):
        build_index.construir(tamanho, docs)
    print("\n######## Consulta de exemplo ########", flush=True)
    from src.rag import buscar
    print(buscar("Quais cuidados de uso ajudam a reduzir o consumo de energia com o ar-condicionado?"))
    print("\n######## Avaliação RAGAS ########", flush=True)
    from evals.ragas_eval import executar_avaliacao
    executar_avaliacao()
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


Writing /content/CKP02_DocMind_RAG_Elytra_Energy/scripts/executar_tudo.py


In [19]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/evals/__init__.py

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/evals/__init__.py


In [20]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/evals/ragas_eval.py
"""Avaliação RAGAS: faithfulness + answer_relevancy, 5 perguntas × 2 configurações de chunking (512 e 1024).

Gera:
- reports/ragas_por_pergunta.csv  (tabela por pergunta e por configuração — exigência do PDF)
- reports/ragas_resumo.md         (médias, tabela por pergunta e a escolha justificada pelos dados)
Os números vêm do RAGAS; nada é preenchido manualmente.
"""
from __future__ import annotations

import csv
import json
import math
import statistics

from src.chunking import config_chunking, dividir_documentos
from src.config import CHUNK_SIZES, QUESTIONS_FILE, REPORTS_DIR
EVALUATION_CHUNK_SIZES = (512, 1024)
from src.embeddings import criar_embeddings
from src.generator import criar_modelo_chat
from src.loaders import carregar_documentos
from src.rag import RAGPipeline
from src.vectorstore import carregar_vectorstore, construir_vectorstore


def carregar_perguntas() -> list[dict]:
    perguntas = json.loads(QUESTIONS_FILE.read_text(encoding="utf-8"))
    if len(perguntas) < 5:
        raise ValueError("O PDF exige no mínimo 5 perguntas em evals/perguntas.json.")
    return perguntas


def _metricas():
    """Instancia as métricas compatíveis com a versão instalada do RAGAS."""
    try:
        from ragas.metrics import Faithfulness, ResponseRelevancy
        return [Faithfulness(), ResponseRelevancy()]
    except ImportError:  # versões antigas
        from ragas.metrics import answer_relevancy, faithfulness
        return [faithfulness, answer_relevancy]


def _valido(valor) -> bool:
    try:
        return valor is not None and math.isfinite(float(valor))
    except (TypeError, ValueError):
        return False


def _rodar_ragas(amostras: list[dict]):
    from ragas import EvaluationDataset, evaluate
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.llms import LangchainLLMWrapper
    from ragas.run_config import RunConfig

    dataset = EvaluationDataset.from_list(amostras)
    resultado = evaluate(
        dataset,
        metrics=_metricas(),
        llm=LangchainLLMWrapper(criar_modelo_chat()),
        embeddings=LangchainEmbeddingsWrapper(criar_embeddings()),
        run_config=RunConfig(timeout=120, max_retries=2, max_workers=2),
        raise_exceptions=False,
        show_progress=True,
    )
    return resultado.to_pandas().to_dict(orient="records")


def avaliar_configuracao(chunk_size: int, perguntas: list[dict]) -> list[dict]:
    """Roda o pipeline real nas perguntas e mede as métricas dessa configuração."""
    pipeline = RAGPipeline(chunk_size=chunk_size)
    amostras = []
    for item in perguntas:
        resposta = pipeline.consultar(item["question"])
        amostras.append({
            "user_input": item["question"],
            "response": resposta.resposta,
            "retrieved_contexts": resposta.contextos,  # exatamente o que o LLM recebeu
            "reference": item["ground_truth"],
        })
    registros = _rodar_ragas(amostras)
    # Reavalia uma vez as linhas que voltaram sem faithfulness (JSON inválido do LLM etc.).
    pendentes = [i for i, r in enumerate(registros) if not _valido(r.get("faithfulness"))]
    if pendentes:
        print(f"  [aviso] {len(pendentes)} linha(s) sem faithfulness; reavaliando uma vez.")
        novos = _rodar_ragas([amostras[i] for i in pendentes])
        for i, novo in zip(pendentes, novos):
            for campo in ("faithfulness", "answer_relevancy"):
                if _valido(novo.get(campo)):
                    registros[i][campo] = novo[campo]
    if not registros:
        raise RuntimeError(f"RAGAS não retornou linhas para chunk_size={chunk_size}.")
    return [{
        "chunk_size": chunk_size,
        "overlap": config_chunking(chunk_size).chunk_overlap,
        "question": perguntas[i]["question"],
        "faithfulness": r.get("faithfulness"),
        "answer_relevancy": r.get("answer_relevancy"),
    } for i, r in enumerate(registros)]


def _media(valores: list[float]):
    return statistics.mean(valores) if valores else None


def _fmt(valor) -> str:
    return f"{float(valor):.3f}" if _valido(valor) else "N/D"


def _garantir_indices(documentos) -> None:
    """Cria as coleções que ainda não existem (a avaliação funciona mesmo sem pré-indexar)."""
    for tamanho in EVALUATION_CHUNK_SIZES:
        try:
            carregar_vectorstore(tamanho)
        except (FileNotFoundError, ValueError):
            construir_vectorstore(dividir_documentos(documentos, tamanho), tamanho, substituir=True)


def executar_avaliacao() -> list[dict]:
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    perguntas = carregar_perguntas()
    _garantir_indices(carregar_documentos())

    linhas: list[dict] = []
    for tamanho in EVALUATION_CHUNK_SIZES:
        print(f"\n[Avaliando] chunk_size={tamanho} | {len(perguntas)} perguntas")
        linhas.extend(avaliar_configuracao(tamanho, perguntas))

    caminho_csv = REPORTS_DIR / "ragas_por_pergunta.csv"
    with caminho_csv.open("w", newline="", encoding="utf-8-sig") as arquivo:
        gravador = csv.DictWriter(arquivo, fieldnames=["chunk_size", "overlap", "question",
                                                       "faithfulness", "answer_relevancy"])
        gravador.writeheader()
        gravador.writerows(linhas)

    medias = []
    for tamanho in EVALUATION_CHUNK_SIZES:
        grupo = [l for l in linhas if l["chunk_size"] == tamanho]
        faith = [float(l["faithfulness"]) for l in grupo if _valido(l["faithfulness"])]
        relev = [float(l["answer_relevancy"]) for l in grupo if _valido(l["answer_relevancy"])]
        medias.append({"chunk_size": tamanho, "faith": _media(faith), "relev": _media(relev), "n": len(faith)})

    validos = [m for m in medias if m["faith"] is not None]
    escolhido = max(validos, key=lambda m: (m["faith"], m["relev"] or -1)) if validos else None

    md = ["# Relatório RAGAS — Elytra Energy (CKP02)", "",
          f"Perguntas por configuração: {len(perguntas)}. Valores gerados pelo RAGAS nesta execução.", "",
          "## Médias por configuração de chunking", "",
          "| Chunk size | Overlap | Faithfulness média | Answer relevancy média | Perguntas com faithfulness |",
          "|---:|---:|---:|---:|---:|"]
    for m in medias:
        md.append(f"| {m['chunk_size']} | {config_chunking(m['chunk_size']).chunk_overlap} "
                  f"| {_fmt(m['faith'])} | {_fmt(m['relev'])} | {m['n']}/{len(perguntas)} |")
    md += ["", "## Resultado por pergunta", "",
           "| Chunk | # | Faithfulness | Answer relevancy | Pergunta |", "|---:|---:|---:|---:|---|"]
    for tamanho in EVALUATION_CHUNK_SIZES:
        for i, l in enumerate([x for x in linhas if x["chunk_size"] == tamanho], start=1):
            md.append(f"| {tamanho} | {i} | {_fmt(l['faithfulness'])} | {_fmt(l['answer_relevancy'])} "
                      f"| {l['question']} |")
    md += ["", "## Configuração escolhida", ""]
    if escolhido:
        f = escolhido["faith"]
        md.append(f"Pelos dados desta execução, a melhor configuração é **chunk_size={escolhido['chunk_size']}** "
                  f"(faithfulness média {f:.3f}; answer relevancy média {_fmt(escolhido['relev'])}). "
                  "O critério foi maior faithfulness média, com answer relevancy como desempate.")
        md.append("")
        if f < 0.5:
            md.append("**Atenção:** faithfulness abaixo de 0,5 indica alucinação significativa; itere no chunking antes de entregar.")
        elif f < 0.7:
            md.append("**Atenção:** faithfulness média abaixo da meta de aprovação (0,7); revise chunks, top-k e prompt.")
        elif f < 0.9:
            md.append("Faithfulness média acima da meta de aprovação (0,7), abaixo da zona ideal (0,9).")
        else:
            md.append("Faithfulness média na zona ideal (≥ 0,9).")
    else:
        md.append("O RAGAS não devolveu métricas válidas; não escolha uma configuração até corrigir a avaliação.")
    (REPORTS_DIR / "ragas_resumo.md").write_text("\n".join(md) + "\n", encoding="utf-8")

    print(f"\nCSV: {caminho_csv}\nResumo: {REPORTS_DIR / 'ragas_resumo.md'}")
    for m in medias:
        print(f"chunk {m['chunk_size']}: faithfulness={_fmt(m['faith'])} | answer_relevancy={_fmt(m['relev'])}")
    return linhas


def main() -> int:
    executar_avaliacao()
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/evals/ragas_eval.py


In [21]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/evals/perguntas.json
[
  {
    "question": "Quais sistemas e equipamentos são tratados no Guia para Eficiência Energética nas Edificações Públicas do MME?",
    "ground_truth": "O guia trata de sistemas como ar-condicionado, iluminação, envoltória e aspectos construtivos, aquecimento solar de água, elevadores e sistemas de supervisão, controle e aquisição de dados."
  },
  {
    "question": "O que a etiqueta de eficiência energética do Inmetro (PBE) informa sobre os condicionadores de ar?",
    "ground_truth": "A tabela do Programa Brasileiro de Etiquetagem do Inmetro apresenta os modelos de condicionadores de ar com seus dados de desempenho e a classificação de eficiência energética."
  },
  {
    "question": "Que informações constam na tabela do Inmetro para refrigeradores e como usá-las ao comparar modelos?",
    "ground_truth": "A tabela do Inmetro lista modelos de refrigeradores com dados de consumo e classificação de eficiência, que permitem comparar modelos de mesma categoria."
  },
  {
    "question": "Como a EPE acompanha o progresso da eficiência energética no Brasil no Atlas da Eficiência Energética?",
    "ground_truth": "O Atlas da Eficiência Energética Brasil monitora o progresso por meio de indicadores de eficiência energética, incluindo análises por setor."
  },
  {
    "question": "O que o Inmetro informa sobre ventiladores de mesa, coluna e circuladores de ar na tabela de eficiência energética?",
    "ground_truth": "A tabela do Inmetro apresenta os ventiladores de mesa, coluna e circuladores de ar registrados no programa de etiquetagem, com seus dados de desempenho."
  }
]


Writing /content/CKP02_DocMind_RAG_Elytra_Energy/evals/perguntas.json


In [22]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/data/fontes.json
{
  "documentos": {
    "01_inmetro_condicionadores_de_ar_2025.pdf": {
      "fonte": "https://www.gov.br/inmetro/pt-br/assuntos/regulamentacao/avaliacao-da-conformidade/programa-brasileiro-de-etiquetagem/tabelas-de-eficiencia-energetica/condicionadores-de-ar/condicionadores-de-ar-2025.pdf/view",
      "data": "2025",
      "categoria": "Climatização",
      "tipo": "PDF",
      "instituicao": "INMETRO",
      "max_paginas": 40
    },
    "02_inmetro_refrigeradores_2025.pdf": {
      "fonte": "https://www.gov.br/inmetro/pt-br/assuntos/regulamentacao/avaliacao-da-conformidade/programa-brasileiro-de-etiquetagem/tabelas-de-eficiencia-energetica/refrigeradores-frigobares-combinados-combinados-frost-free/refrigerador.pdf/view",
      "data": "2025",
      "categoria": "Refrigeração",
      "tipo": "PDF",
      "instituicao": "INMETRO",
      "max_paginas": 40
    },
    "03_inmetro_ventiladores_2025.pdf": {
      "fonte": "https://www.gov.br/inmetro/pt-br/assuntos/regulamentacao/avaliacao-da-conformidade/programa-brasileiro-de-etiquetagem/tabelas-de-eficiencia-energetica/ventiladores/ventiladores-de-mesa-coluna-e-circuladores-de-ar.pdf/view",
      "data": "2025",
      "categoria": "Ventilação",
      "tipo": "PDF",
      "instituicao": "INMETRO",
      "max_paginas": 40
    },
    "04_epe_atlas_eficiencia_energetica_2024.pdf": {
      "fonte": "https://www.epe.gov.br/pt/publicacoes-dados-abertos/publicacoes/atlas-de-eficiencia-energetica-brasil-2024",
      "data": "2024",
      "categoria": "Indicadores energéticos",
      "tipo": "PDF",
      "instituicao": "EPE",
      "max_paginas": 120
    },
    "05_mme_guia_eficiencia_energetica_edificios_publicos.pdf": {
      "fonte": "https://www.gov.br/mme/pt-br/centrais-de-conteudo/publicacoes/eficiencia-energetica/guia-de-eficiencia-energetica-em-edificios-publicos.pdf/@@download/file",
      "data": "2015",
      "categoria": "Eficiência energética em edificações",
      "tipo": "PDF",
      "instituicao": "MME/CEPEL",
      "max_paginas": 60
    }
  }
}


Writing /content/CKP02_DocMind_RAG_Elytra_Energy/data/fontes.json


In [23]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/prompts/system_prompt.md
Você é o Elytra Energy, um assistente técnico em eficiência energética residencial brasileiro.

Regras obrigatórias:
1. Responda em português brasileiro, com clareza e objetividade.
2. Use SOMENTE o conteúdo dos trechos recuperados como evidência factual. Se os trechos não bastarem, diga exatamente: "A base consultada não permite confirmar essa informação." e explique o que falta.
3. Não invente consumos, potências, classificações, normas, valores monetários, datas ou recomendações atribuídas às fontes. Não use conhecimento externo aos trechos.
4. Cada afirmação factual deve terminar com a citação do trecho de onde veio, no formato [Trecho N] (por exemplo: "Limpe os filtros com frequência [Trecho 2]."). Use apenas números de trechos que existem no contexto.
5. Diferencie dados oficiais de inferências simples. Se fizer uma conta, mostre os dados e a fórmula usados.
6. Para comparar equipamentos, use apenas os valores e categorias presentes nos trechos; não conclua que um equipamento é melhor sem evidência recuperada.
7. Seja direto: no máximo 8 frases curtas ou uma lista curta. Não escreva seção de fontes: o sistema anexa arquivo, página, chunk e URL automaticamente.

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/prompts/system_prompt.md


In [24]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/main.py
"""Interface Gradio do DocMind RAG — Elytra Energy (diferencial do PDF)."""
from __future__ import annotations

import gradio as gr

from src.config import CATEGORIAS, DEFAULT_CHUNK_SIZE
from src.rag import obter_pipeline


def criar_interface() -> gr.Blocks:
    pipeline = obter_pipeline(DEFAULT_CHUNK_SIZE)

    def responder(mensagem: str, historico, categoria: str) -> str:
        del historico  # o RAG responde pela base documental, não por memória de conversa
        filtro = None if not categoria or categoria == "Todas" else categoria
        try:
            return pipeline.consultar(mensagem, categoria=filtro).para_texto()
        except Exception as exc:  # mostra o erro na tela em vez de travar a interface
            return f"Não consegui responder: {exc}"

    with gr.Blocks(title="Elytra Energy · DocMind RAG") as demo:
        gr.Markdown(
            "# Elytra Energy — DocMind RAG\n"
            "Assistente de eficiência energética residencial baseado em documentos oficiais "
            "(INMETRO, EPE e MME/CEPEL). Cada resposta cita arquivo, página, chunk e link de origem.\n\n"
            f"Índice ativo: `chunk_size={DEFAULT_CHUNK_SIZE}` · chat: `gemma4:cloud` (Ollama Cloud) "
            "· embeddings: `nomic-embed-text`"
        )
        gr.ChatInterface(
            fn=responder,
            additional_inputs=[gr.Dropdown(choices=["Todas"] + CATEGORIAS, value="Todas",
                                           label="Filtrar por categoria (metadata filtering)")],
            examples=[
                ["Quais sistemas são abordados no Guia para Eficiência Energética nas Edificações Públicas?", "Eficiência energética em edificações"],
                ["O que a tabela do Inmetro informa sobre ventiladores?", "Ventilação"],
                ["Como a EPE acompanha a eficiência energética no Brasil?", "Indicadores energéticos"],
            ],
        )
    return demo


if __name__ == "__main__":
    criar_interface().launch()


Writing /content/CKP02_DocMind_RAG_Elytra_Energy/main.py


In [25]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/tests/__init__.py

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/tests/__init__.py


In [26]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/tests/test_chunking.py
"""Testes do chunking exigido pelo PDF."""
from langchain_core.documents import Document

from src.chunking import SEPARADORES, config_chunking, criar_splitter, dividir_documentos


def test_separadores_exigidos_pelo_enunciado():
    assert SEPARADORES == ["\n\n", "\n", ". ", " ", ""]


def test_overlap_fica_entre_dez_e_quinze_porcento():
    for tamanho in (256, 512, 1024):
        c = config_chunking(tamanho)
        assert 0.10 <= c.chunk_overlap / c.chunk_size <= 0.15


def test_divisao_preserva_proveniencia_e_gera_chunk_id():
    texto = "Eficiência energética residencial. Compare o consumo informado pelo fabricante.\n\n" * 30
    origem = Document(page_content=texto, metadata={
        "arquivo": "teste.pdf", "pagina": 4, "fonte": "https://exemplo.gov.br/t.pdf",
        "categoria": "Residencial", "tipo": "PDF", "data": "2025"})
    chunks = dividir_documentos([origem], 256)
    assert len(chunks) > 1
    assert all(c.metadata["arquivo"] == "teste.pdf" and c.metadata["pagina"] == 4 for c in chunks)
    assert chunks[0].metadata["chunk_id"].startswith("teste.pdf:4:")


def test_splitter_usa_tamanho_configurado():
    assert criar_splitter(512).chunk_size == 512

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/tests/test_chunking.py


In [27]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/tests/test_schemas.py
"""Testes dos schemas Pydantic."""
import pytest
from pydantic import ValidationError

from src.schemas import DocumentMetadata, DocumentRecord, RAGResponse, SourceCitation


def _meta():
    return DocumentMetadata(arquivo="a.pdf", tipo="PDF", categoria="Residencial",
                            data="2025", fonte="https://exemplo.gov.br/a.pdf")


def test_documento_valida_conteudo():
    assert DocumentRecord(conteudo="  texto  ", metadata=_meta()).conteudo == "texto"


def test_documento_rejeita_conteudo_vazio():
    with pytest.raises(ValidationError):
        DocumentRecord(conteudo="   ", metadata=_meta())


def test_resposta_mostra_arquivo_pagina_chunk_e_url():
    r = RAGResponse(consulta="Como economizar?", resposta="Limpe os filtros [Trecho 1].",
                    fontes=[SourceCitation(trecho_numero=1, arquivo="dicas.pdf", pagina=3,
                                           fonte="https://exemplo.gov.br/dicas.pdf",
                                           chunk_id="dicas.pdf:3:0:512", trecho="texto do chunk")])
    texto = r.para_texto()
    for esperado in ("dicas.pdf", "página 3", "dicas.pdf:3:0:512", "https://exemplo.gov.br/dicas.pdf"):
        assert esperado in texto

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/tests/test_schemas.py


In [28]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/tests/test_retriever.py
"""Testes do metadata filtering (where) e do reranking."""
import pytest
from langchain_core.documents import Document

from src import retriever
from src.reranker import reranquear


def test_filtro_por_categoria():
    assert retriever.montar_filtro("Climatização") == {"categoria": "Climatização"}


def test_filtro_todas_nao_filtra():
    assert retriever.montar_filtro("Todas") is None and retriever.montar_filtro(None) is None


def test_filtro_combinado_usa_and():
    f = retriever.montar_filtro("Climatização", "INMETRO")
    assert f == {"$and": [{"categoria": "Climatização"}, {"instituicao": "INMETRO"}]}


def test_categoria_invalida_gera_erro():
    with pytest.raises(ValueError):
        retriever.montar_filtro("Inexistente")


def test_reranking_desligado_mantem_ordem():
    docs = [Document(page_content="a"), Document(page_content="b")]
    assert reranquear("q", docs, habilitado=False) == docs

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/tests/test_retriever.py


In [29]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/tests/test_generator_embeddings.py
"""Testes do nome do modelo de chat (gemma4:cloud) e da escolha Cloud/local dos embeddings."""
from types import SimpleNamespace

from src import embeddings, generator


class _Resp:
    def __init__(self, status=200, corpo=None):
        self.status_code, self._corpo = status, corpo or {}

    def json(self):
        return self._corpo

    def raise_for_status(self):
        if self.status_code >= 400:
            raise RuntimeError(self.status_code)


def test_resolve_gemma4_cloud_para_nome_da_api(monkeypatch):
    generator.resolver_nome_modelo.cache_clear()
    monkeypatch.setattr(generator, "OLLAMA_API_KEY", "chave-de-teste")
    monkeypatch.setattr(generator, "CHAT_MODEL", "gemma4:cloud")
    monkeypatch.setattr(generator.requests, "get",
                        lambda *a, **k: _Resp(200, {"models": [{"name": "gpt-oss:120b"}, {"name": "gemma4:31b"}]}))
    assert generator.resolver_nome_modelo() == "gemma4:31b"
    generator.resolver_nome_modelo.cache_clear()


def test_embeddings_cai_para_local_quando_cloud_da_401(monkeypatch):
    embeddings.modo_embeddings_efetivo.cache_clear()
    monkeypatch.setattr(embeddings, "EMBEDDING_MODE", "auto")
    monkeypatch.setattr(embeddings, "OLLAMA_API_KEY", "chave-de-teste")
    monkeypatch.setattr(embeddings.requests, "post", lambda *a, **k: _Resp(401))
    monkeypatch.setattr(embeddings.requests, "get",
                        lambda *a, **k: _Resp(200, {"models": [{"name": "nomic-embed-text:latest"}]}))
    assert embeddings.modo_embeddings_efetivo() == "local"
    embeddings.modo_embeddings_efetivo.cache_clear()


def test_embeddings_usa_cloud_quando_ele_responde(monkeypatch):
    embeddings.modo_embeddings_efetivo.cache_clear()
    monkeypatch.setattr(embeddings, "EMBEDDING_MODE", "auto")
    monkeypatch.setattr(embeddings, "OLLAMA_API_KEY", "chave-de-teste")
    monkeypatch.setattr(embeddings.requests, "post", lambda *a, **k: _Resp(200, {"embeddings": [[0.1, 0.2]]}))
    assert embeddings.modo_embeddings_efetivo() == "cloud"
    embeddings.modo_embeddings_efetivo.cache_clear()


def test_adaptador_usa_modelo_aprovado(monkeypatch):
    embeddings.modo_embeddings_efetivo.cache_clear()
    capturado = {}
    monkeypatch.setattr(embeddings, "modo_embeddings_efetivo", lambda: "local")
    monkeypatch.setattr(embeddings, "OllamaEmbeddings", lambda **kw: capturado.update(kw) or SimpleNamespace(**kw))
    embeddings.criar_embeddings()
    assert capturado["model"] == "nomic-embed-text"

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/tests/test_generator_embeddings.py


In [30]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/tests/test_vectorstore_compat.py
from types import SimpleNamespace

from src.vectorstore import _nomes_colecoes


def test_aceita_nomes_como_texto():
    class C:
        def list_collections(self):
            return ["a", "b"]
    assert _nomes_colecoes(C()) == {"a", "b"}


def test_aceita_objetos_com_nome():
    class C:
        def list_collections(self):
            return [SimpleNamespace(name="x")]
    assert _nomes_colecoes(C()) == {"x"}

Writing /content/CKP02_DocMind_RAG_Elytra_Energy/tests/test_vectorstore_compat.py


In [31]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/README.md
# CKP02 — DocMind RAG | Elytra Energy

**FIAP · Prompt Engineering and Artificial Intelligence · 2º semestre de 2026**
**Integrantes:** Kaique Da Silva Assis (RM 572718) · Andre Debiazzi (RM 569062) · Vinicius Cristal de Oliveira (RM 572049)

## Objetivo
Pipeline RAG sobre fontes oficiais de eficiência energética, com `load → split → embed → store → retrieve → generate`, ChromaDB, citações por arquivo/página, comparação de chunking e avaliação RAGAS.

## Como executar no Google Colab
1. Abra o notebook CKP02 no Colab.
2. Em *Ambiente de execução → Alterar tipo de ambiente de execução*, selecione T4 GPU caso esteja disponível.
3. Crie o Secret `OLLAMA_API_KEY` no painel Secrets e habilite o acesso ao notebook.
4. Execute as células na ordem. A primeira célula instala dependências, inicia o Ollama e monta o cache no Google Drive.
5. A célula de fontes baixa (ou restaura do Drive) os cinco PDFs oficiais.
6. Execute a comparação, consulta RAG, testes e avaliação RAGAS. Ao final, baixe o ZIP de entrega e exporte também o notebook executado (`File → Download → Download .ipynb`).

## Fontes e documentos
A lista completa de fontes está em `data/fontes.json`; os PDFs ficam em `data/originals/`. Para incluir um documento novo, coloque o PDF/TXT/Markdown em `data/originals/` e adicione uma entrada em `data/fontes.json` com a URL oficial, instituição, data, categoria e tipo.

## Modo rápido e retomada
Por padrão, o notebook indexa as primeiras 15 páginas de cada PDF para acelerar a execução. Os arquivos PDF completos são preservados no pacote. Se precisar indexar todas as páginas, mude `RAG_MAX_PAGES_PER_PDF` para `0` na célula de preparação. Embeddings calculados são guardados em cache por lote no Google Drive, permitindo retomar lotes já calculados após uma desconexão. A comparação principal utiliza `chunk_size=512` e `chunk_size=1024`, duas estratégias distintas.

## Arquitetura
- `src/`: loaders, chunking, embeddings, vectorstore, retriever, reranker, generator e pipeline RAG.
- `scripts/`: download das fontes, diagnóstico e construção de índices.
- `evals/`: perguntas de referência e métricas RAGAS (`faithfulness` e `answer_relevancy`).
- `tests/`: testes automatizados.
- `prompts/system_prompt.md`: regras de resposta fundamentada com citação.
- `reports/`: resultados reais da avaliação, gerados ao executar o RAGAS.

A chave da API deve permanecer nos Secrets do Colab ou em `.env` local; nunca publique uma chave real no repositório.


Writing /content/CKP02_DocMind_RAG_Elytra_Energy/README.md


In [32]:
%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/requirements.txt
langchain-core
langchain-text-splitters
langchain-ollama
langchain-chroma
langchain-openai
chromadb
numpy
pydantic
python-dotenv
pypdf
requests
gradio
ragas>=0.2.10,<0.4
datasets
pytest
# Opcional para ativar o reranker: sentence-transformers


Writing /content/CKP02_DocMind_RAG_Elytra_Energy/requirements.txt


## 2. Diagnóstico (chave, chat `gemma4:cloud` e embeddings `nomic-embed-text`)

In [33]:
import os, sys
os.chdir("/content/CKP02_DocMind_RAG_Elytra_Energy")
sys.path.insert(0, "/content/CKP02_DocMind_RAG_Elytra_Energy")
from scripts.diagnostico import main as diagnosticar
if diagnosticar() != 0:
    raise RuntimeError("Diagnóstico falhou: corrija os itens acima antes de continuar.")

=== Diagnóstico DocMind RAG ===
Chave OLLAMA_API_KEY: CARREGADA
Chat: gemma4:cloud em https://ollama.com | Embeddings: nomic-embed-text (modo auto)
[embeddings] Ollama Cloud não atendeu embeddings (HTTP 401); usando Ollama do runtime.
[OK] Embedding de 768 dimensões (origem: local).
[OK] Chat 'gemma4:31b' respondeu: conexão OK

Diagnóstico concluído com sucesso.


## 3. Base de conhecimento real (≥ 5 documentos oficiais com fonte citada)
Os PDFs são baixados dos sites oficiais do INMETRO, EPE e Ministério de Minas e Energia (MME/CEPEL). As URLs de origem estão em `data/fontes.json`.

In [34]:
import json, os, shutil
from pathlib import Path
from scripts import baixar_fontes
from src.config import DATA_DIR

# Reaproveita os PDFs já baixados mesmo após uma desconexão do runtime.
cache_docs = Path(os.environ["RAG_PERSISTENT_DOCS_DIR"])
cache_docs.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
for pdf in cache_docs.glob("*.pdf"):
    destino = DATA_DIR / pdf.name
    if not destino.exists() or destino.stat().st_size != pdf.stat().st_size:
        shutil.copy2(pdf, destino)

if baixar_fontes.main() != 0:
    raise RuntimeError(
        "Algum PDF não baixou. Verifique a pasta cache/originals ou baixe o PDF no navegador e envie para data/originals/.")

# Persiste os documentos recém-baixados para que não sejam baixados novamente.
for pdf in DATA_DIR.glob("*.pdf"):
    shutil.copy2(pdf, cache_docs / pdf.name)

fontes = json.load(open("data/fontes.json", encoding="utf-8"))["documentos"]
for nome, info in fontes.items():
    print(f"- {nome} | {info['instituicao']} | {info['categoria']} | {info['fonte']}")
print("PDFs persistidos no Google Drive:", cache_docs)


[BAIXANDO] 01_inmetro_condicionadores_de_ar_2025.pdf
[OK] 01_inmetro_condicionadores_de_ar_2025.pdf (3,725,324 bytes)
[BAIXANDO] 02_inmetro_refrigeradores_2025.pdf
[OK] 02_inmetro_refrigeradores_2025.pdf (2,817,151 bytes)
[BAIXANDO] 03_inmetro_ventiladores_2025.pdf
[OK] 03_inmetro_ventiladores_2025.pdf (3,140,565 bytes)
[BAIXANDO] 04_epe_atlas_eficiencia_energetica_2024.pdf
[OK] 04_epe_atlas_eficiencia_energetica_2024.pdf (3,440,858 bytes)
[BAIXANDO] 05_mme_guia_eficiencia_energetica_edificios_publicos.pdf
[OK] 05_mme_guia_eficiencia_energetica_edificios_publicos.pdf (7,100,471 bytes)

Documentos válidos: 5/5
Base pronta em: /content/CKP02_DocMind_RAG_Elytra_Energy/data/originals
- 01_inmetro_condicionadores_de_ar_2025.pdf | INMETRO | Climatização | https://www.gov.br/inmetro/pt-br/assuntos/regulamentacao/avaliacao-da-conformidade/programa-brasileiro-de-etiquetagem/tabelas-de-eficiencia-energetica/condicionadores-de-ar/condicionadores-de-ar-2025.pdf/view
- 02_inmetro_refrigeradores_202

## 4. Pipeline `load → split → embed → store` (check-in da Aula 05)
Documentos carregados, chunks com `RecursiveCharacterTextSplitter`, embeddings `nomic-embed-text`, coleção no ChromaDB
e uma busca semântica retornando resultado.

In [35]:
from src.loaders import carregar_documentos
from src.chunking import dividir_documentos, config_chunking
from src.embeddings import criar_embeddings, modo_embeddings_efetivo
from src.vectorstore import construir_vectorstore, carregar_vectorstore

print("Carregando documentos (modo rápido: até 15 páginas por PDF)...", flush=True)
docs = carregar_documentos()
print("Páginas carregadas:", len(docs), "| exemplo metadata:", docs[0].metadata, flush=True)
chunks_512 = dividir_documentos(docs, 512)
print("Chunks (512):", len(chunks_512), "| overlap:", config_chunking(512).chunk_overlap, flush=True)
emb = criar_embeddings()
print("Embeddings via:", modo_embeddings_efetivo(), "| dimensões:", len(emb.embed_query("eficiência energética residencial")), flush=True)

# Reaproveita o índice se já estiver completo. Se a execução for interrompida,
# o próximo runtime recupera os vetores do cache persistente no Drive.
try:
    store_512 = carregar_vectorstore(512)
    if store_512._collection.count() != len(chunks_512):
        raise ValueError("Índice 512 incompleto")
    print("[REUTILIZADO] Índice 512 já está completo.", flush=True)
except (FileNotFoundError, ValueError):
    print("[INÍCIO] Construindo índice 512...", flush=True)
    store_512 = construir_vectorstore(chunks_512, 512, substituir=True)

print("Coleção:", store_512._collection.name, "| registros:", store_512._collection.count(), flush=True)
for r in store_512.similarity_search("dicas para economizar energia com ar-condicionado", k=3):
    print("-", r.metadata["arquivo"], "p.", r.metadata["pagina"], "|", r.page_content[:120].replace("\n", " "))

Carregando documentos (modo rápido: até 15 páginas por PDF)...
Páginas carregadas: 71 | exemplo metadata: {'arquivo': '01_inmetro_condicionadores_de_ar_2025.pdf', 'tipo': 'PDF', 'categoria': 'Climatização', 'data': '2025', 'fonte': 'https://www.gov.br/inmetro/pt-br/assuntos/regulamentacao/avaliacao-da-conformidade/programa-brasileiro-de-etiquetagem/tabelas-de-eficiencia-energetica/condicionadores-de-ar/condicionadores-de-ar-2025.pdf/view', 'instituicao': 'INMETRO', 'pagina': 1}
Chunks (512): 1387 | overlap: 61
Embeddings via: local | dimensões: 768
[INÍCIO] Construindo índice 512...
  [eficiencia_energetica_residencial_512] 96/1387 chunks | lote 1/15 | cache usados: 0/1 | 0.3 min | embedding calculado
  [eficiencia_energetica_residencial_512] 192/1387 chunks | lote 2/15 | cache usados: 0/2 | 0.3 min | embedding calculado
  [eficiencia_energetica_residencial_512] 288/1387 chunks | lote 3/15 | cache usados: 0/3 | 0.3 min | embedding calculado
  [eficiencia_energetica_residencial_512] 384

## 5. Comparação de estratégias de chunking (512 e 1024; overlap de 12%)

In [36]:
from src.vectorstore import construir_vectorstore, carregar_vectorstore

# O enunciado pede pelo menos duas estratégias. 512 já é o índice principal;
# 1024 completa a comparação sem pagar o custo extra da configuração 256.
TAMANHOS_COMPARACAO = (512, 1024)
contagens = {}
for tamanho in TAMANHOS_COMPARACAO:
    chunks = dividir_documentos(docs, tamanho)
    contagens[tamanho] = len(chunks)
    try:
        store = carregar_vectorstore(tamanho)
        if store._collection.count() != len(chunks):
            raise ValueError("Índice incompleto")
        print(f"[REUTILIZADO] chunk_size={tamanho} | índice completo", flush=True)
    except (FileNotFoundError, ValueError):
        print(f"[INÍCIO] chunk_size={tamanho} | {len(chunks)} chunks", flush=True)
        store = construir_vectorstore(chunks, tamanho, substituir=True)
    print(f"chunk_size={tamanho} | overlap={config_chunking(tamanho).chunk_overlap} | chunks={len(chunks)} | registros={store._collection.count()}", flush=True)
contagens


[REUTILIZADO] chunk_size=512 | índice completo
chunk_size=512 | overlap=61 | chunks=1387 | registros=1387
[INÍCIO] chunk_size=1024 | 671 chunks
  [eficiencia_energetica_residencial_1024] 96/671 chunks | lote 1/7 | cache usados: 0/1 | 0.0 min | embedding calculado
  [eficiencia_energetica_residencial_1024] 192/671 chunks | lote 2/7 | cache usados: 0/2 | 0.1 min | embedding calculado
  [eficiencia_energetica_residencial_1024] 288/671 chunks | lote 3/7 | cache usados: 0/3 | 0.1 min | embedding calculado
  [eficiencia_energetica_residencial_1024] 384/671 chunks | lote 4/7 | cache usados: 0/4 | 0.1 min | embedding calculado
  [eficiencia_energetica_residencial_1024] 480/671 chunks | lote 5/7 | cache usados: 0/5 | 0.1 min | embedding calculado
  [eficiencia_energetica_residencial_1024] 576/671 chunks | lote 6/7 | cache usados: 0/6 | 0.2 min | embedding calculado
  [eficiencia_energetica_residencial_1024] 671/671 chunks | lote 7/7 | cache usados: 0/7 | 0.2 min | embedding calculado
[concluído

{512: 1387, 1024: 671}

## 6. Consulta RAG: `retrieve → generate` com citação do chunk de origem

In [37]:
from src.rag import RAGPipeline
pipeline = RAGPipeline(chunk_size=512)
resposta = pipeline.consultar("Quais cuidados de uso ajudam a reduzir o consumo de energia com o ar-condicionado em casa?")
print(resposta.para_texto())

A base consultada não permite confirmar essa informação. Os trechos fornecidos tratam de legislações, programas de etiquetagem do Inmetro e noções gerais de eficiência em edifícios públicos, mas não listam cuidados de uso específicos para reduzir o consumo de ar-condicionado residencial.

**Fontes consultadas**
- [Trecho 1] `05_mme_guia_eficiencia_energetica_edificios_publicos.pdf`, página 11 (chunk `05_mme_guia_eficiencia_energetica_edificios_publicos.pdf:11:1369:512`) — Eficiência energética em edificações — https://www.gov.br/mme/pt-br/centrais-de-conteudo/publicacoes/eficiencia-energetica/guia-de-eficiencia-energetica-em-edificios-publicos.pdf/@@download/file
  > As características dos prédios levantadas nas visitas (construtivas, da envoltória da edificação, operacional, perfis de ocupação etc.), confirmaram que há dife…
- [Trecho 2] `05_mme_guia_eficiencia_energetica_edificios_publicos.pdf`, página 9 (chunk `05_mme_guia_eficiencia_energetica_edificios_publicos.pdf:9:1357:512`) — 

### Diferenciais: metadata filtering (`where=`) e reranking

In [38]:
from src.retriever import recuperar
for categoria in ["Residencial", "Ventilação"]:
    achados = recuperar(pipeline.vectorstore, "como usar com eficiência", categoria=categoria, k=3)
    print(categoria, "->", {(d.metadata["arquivo"], d.metadata["pagina"]) for d in achados})
from src.rag import buscar
print(buscar("Como a EPE acompanha o progresso da eficiência energética?")[:700])

Residencial -> set()
Ventilação -> {('03_inmetro_ventiladores_2025.pdf', 1), ('03_inmetro_ventiladores_2025.pdf', 10)}
A base consultada não permite confirmar essa informação, pois os trechos fornecidos não mencionam a EPE (Empresa de Pesquisa Energética) ou como ela especificamente acompanha o progresso da eficiência energética.

**Fontes consultadas**
- [Trecho 1] `05_mme_guia_eficiencia_energetica_edificios_publicos.pdf`, página 7 (chunk `05_mme_guia_eficiencia_energetica_edificios_publicos.pdf:7:1346:512`) — Eficiência energética em edificações — https://www.gov.br/mme/pt-br/centrais-de-conteudo/publicacoes/eficiencia-energetica/guia-de-eficiencia-energetica-em-edificios-publicos.pdf/@@download/file
  > O PBE promove a eficiência energética por meio de etiquetas informativas a respeito do desempenho de m


## 7. Testes automatizados

In [40]:

%%writefile /content/CKP02_DocMind_RAG_Elytra_Energy/tests/test_chunking.py
"""Testes do chunking exigido pelo PDF."""
from langchain_core.documents import Document

from src.chunking import (
    SEPARADORES,
    config_chunking,
    criar_splitter,
    dividir_documentos,
)


def test_separadores_exigidos_pelo_enunciado():
    assert SEPARADORES == ["\n\n", "\n", ". ", " ", ""]


def test_overlap_fica_entre_dez_e_quinze_porcento():
    for tamanho in (256, 512, 1024):
        c = config_chunking(tamanho)
        assert 0.10 <= c.chunk_overlap / c.chunk_size <= 0.15


def test_divisao_preserva_proveniencia_e_gera_chunk_id():
    texto = (
        "Eficiência energética residencial. "
        "Compare o consumo informado pelo fabricante.\n\n"
    ) * 30

    origem = Document(
        page_content=texto,
        metadata={
            "arquivo": "teste.pdf",
            "pagina": 4,
            "fonte": "https://exemplo.gov.br/t.pdf",
            "categoria": "Residencial",
            "tipo": "PDF",
            "data": "2025",
        },
    )

    chunks = dividir_documentos([origem], 256)

    assert len(chunks) > 1
    assert all(
        c.metadata["arquivo"] == "teste.pdf"
        and c.metadata["pagina"] == 4
        for c in chunks
    )
    assert chunks[0].metadata["chunk_id"].startswith(
        "teste.pdf:4:"
    )


def test_splitter_usa_tamanho_configurado():
    splitter = criar_splitter(512)

    # Compatibilidade entre versões do LangChain.
    tamanho = getattr(
        splitter, "chunk_size",
        getattr(splitter, "_chunk_size", None)
    )
    overlap = getattr(
        splitter, "chunk_overlap",
        getattr(splitter, "_chunk_overlap", None)
    )

    assert tamanho == 512
    assert overlap == config_chunking(512).chunk_overlap


Overwriting /content/CKP02_DocMind_RAG_Elytra_Energy/tests/test_chunking.py


In [41]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-1500:])
assert r.returncode == 0, "Há testes falhando; veja a saída acima."

..................                                                       [100%]
18 passed in 3.51s
 


## 8. Avaliação RAGAS — `faithfulness` e `answer_relevancy`
5 perguntas × 2 configurações de chunking (512 e 1024). A tabela por pergunta e as médias são gravadas em `reports/`.
Meta do professor: faithfulness média ≥ 0,7 (zona ideal ≥ 0,9).

In [43]:
%pip install -q --no-deps "langchain-community==0.3.31"

import importlib.metadata

print(
    "langchain-community:",
    importlib.metadata.version("langchain-community")
)

import ragas
print("Importação do RAGAS funcionou!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.7 MB/s eta 0:00:00
langchain-community: 0.3.31
Importação do RAGAS funcionou!


In [44]:
from evals.ragas_eval import executar_avaliacao
resultados = executar_avaliacao()
import pandas as pd
df = pd.read_csv("reports/ragas_por_pergunta.csv")
display(df)
display(df.groupby("chunk_size")[["faithfulness", "answer_relevancy"]].mean().round(3))
print(open("reports/ragas_resumo.md", encoding="utf-8").read())

# Copia os resultados para o Drive; eles sobrevivem à desconexão do runtime.
import os, shutil
from pathlib import Path
pasta_persistente = Path(os.environ["EMBEDDING_CACHE_DIR"]).parent / "reports"
pasta_persistente.mkdir(parents=True, exist_ok=True)
for arquivo in Path("reports").glob("*"):
    if arquivo.is_file():
        shutil.copy2(arquivo, pasta_persistente / arquivo.name)
print("Relatórios também salvos no Google Drive:", pasta_persistente)



[Avaliando] chunk_size=512 | 5 perguntas


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]


[Avaliando] chunk_size=1024 | 5 perguntas


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]


CSV: /content/CKP02_DocMind_RAG_Elytra_Energy/reports/ragas_por_pergunta.csv
Resumo: /content/CKP02_DocMind_RAG_Elytra_Energy/reports/ragas_resumo.md
chunk 512: faithfulness=0.975 | answer_relevancy=0.495
chunk 1024: faithfulness=1.000 | answer_relevancy=0.716


,chunk_size,overlap,question,faithfulness,answer_relevancy
0,512,61,Quais sistemas e equipamentos são tratados no ...,1.000,0.653961
1,512,61,O que a etiqueta de eficiência energética do I...,1.000,0.854305
2,512,61,Que informações constam na tabela do Inmetro p...,1.000,0.000000
3,512,61,Como a EPE acompanha o progresso da eficiência...,1.000,0.967420
4,512,61,O que o Inmetro informa sobre ventiladores de ...,0.875,0.000000
5,1024,123,Quais sistemas e equipamentos são tratados no ...,1.000,0.721877
6,1024,123,O que a etiqueta de eficiência energética do I...,1.000,0.000000
7,1024,123,Que informações constam na tabela do Inmetro p...,1.000,0.963025
8,1024,123,Como a EPE acompanha o progresso da eficiência...,1.000,0.949641
9,1024,123,O que o Inmetro informa sobre ventiladores de ...,1.000,0.944670


,faithfulness,answer_relevancy
chunk_size,,
512,0.975,0.495
1024,1.000,0.716


# Relatório RAGAS — Elytra Energy (CKP02)

Perguntas por configuração: 5. Valores gerados pelo RAGAS nesta execução.

## Médias por configuração de chunking

| Chunk size | Overlap | Faithfulness média | Answer relevancy média | Perguntas com faithfulness |
|---:|---:|---:|---:|---:|
| 512 | 61 | 0.975 | 0.495 | 5/5 |
| 1024 | 123 | 1.000 | 0.716 | 5/5 |

## Resultado por pergunta

| Chunk | # | Faithfulness | Answer relevancy | Pergunta |
|---:|---:|---:|---:|---|
| 512 | 1 | 1.000 | 0.654 | Quais sistemas e equipamentos são tratados no Guia para Eficiência Energética nas Edificações Públicas do MME? |
| 512 | 2 | 1.000 | 0.854 | O que a etiqueta de eficiência energética do Inmetro (PBE) informa sobre os condicionadores de ar? |
| 512 | 3 | 1.000 | 0.000 | Que informações constam na tabela do Inmetro para refrigeradores e como usá-las ao comparar modelos? |
| 512 | 4 | 1.000 | 0.967 | Como a EPE acompanha o progresso da eficiência energética no Brasil no Atlas da Eficiência Energética

## 9. Interface Gradio (fonte citada na resposta)

In [45]:
from main import criar_interface
demo = criar_interface()
try:
    demo.launch(share=True, prevent_thread_lock=True)
except Exception as exc:
    print("Não foi possível abrir o link público do Gradio:", exc)

/usr/local/lib/python3.13/dist-packages/gradio/components/dropdown.py:242: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: Eficiência energética em edificações or set allow_custom_value=True.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/uvicorn/server.py:86: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
Exception in thread Thread-10 (run):
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/server.py", line 86, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop

Não foi possível abrir o link público do Gradio: Cannot find empty port in range: 7860-7959. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.


## 10. Pacote de entrega (projeto completo, documentos e relatórios)
O ZIP inclui o código-fonte, README, requirements, os cinco PDFs da base e relatórios RAGAS. O notebook executado deve ser exportado separadamente pelo menu File → Download → Download .ipynb.


In [46]:
from pathlib import Path
import zipfile
from google.colab import files

ROOT = Path("/content/CKP02_DocMind_RAG_Elytra_Energy")
ARQUIVO = Path("/content/CKP02_DocMind_RAG_Elytra_Energy_ENTREGA.zip")
INCLUIR = ["README.md", "requirements.txt", "main.py", "src", "scripts", "evals", "tests", "prompts", "data", "reports"]
EXCLUIR_PARTES = {"__pycache__", ".git", ".venv", "chroma_db", ".pytest_cache"}

with zipfile.ZipFile(ARQUIVO, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as pacote:
    for item in INCLUIR:
        caminho = ROOT / item
        if not caminho.exists():
            continue
        if caminho.is_file():
            pacote.write(caminho, caminho.relative_to(ROOT))
            continue
        for arquivo in caminho.rglob("*"):
            if not arquivo.is_file() or any(parte in EXCLUIR_PARTES for parte in arquivo.parts):
                continue
            if arquivo.name == ".gitkeep":
                continue
            pacote.write(arquivo, arquivo.relative_to(ROOT))

print("ZIP de entrega criado:", ARQUIVO)
print("Inclui código, README, dependências, documentos oficiais completos e relatórios disponíveis.")
print("O notebook executado deve ser baixado separadamente pelo menu File → Download → Download .ipynb.")
files.download(str(ARQUIVO))

ZIP de entrega criado: /content/CKP02_DocMind_RAG_Elytra_Energy_ENTREGA.zip
Inclui código, README, dependências, documentos oficiais completos e relatórios disponíveis.
O notebook executado deve ser baixado separadamente pelo menu File → Download → Download .ipynb.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Referências
- INMETRO — Tabelas de eficiência energética (PBE): condicionadores de ar, refrigeradores e ventiladores (2025), links em `data/fontes.json`
- EPE — Atlas da Eficiência Energética Brasil 2024
- MME/CEPEL — Guia para Eficiência Energética nas Edificações Públicas (2015)
- Ollama Cloud: https://docs.ollama.com/cloud · API OpenAI-compatível (cloud direto): https://docs.ollama.com/api/openai-compatibility